In [1]:
import yaml

with open("../configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Usage examples
n_factors    = config["models"]["svd"]["n_factors"]
w_svd        = config["models"]["hybrid"]["w_svd"]
matrix_path  = config["artifacts"]["interaction_matrix"]

In [2]:
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
with open("../data/processed/svd_recommender.pkl", "rb") as f:
    svd_rec = pickle.load(f)

with open("../data/processed/cb_recommender.pkl", "rb") as f:
    cb_rec = pickle.load(f)

popularity_df = pd.read_parquet("../data/processed/artist_popularity.parquet")
interactions  = pd.read_parquet("../data/processed/interactions.parquet")

from src.recommenders.popularity import PopularityRecommender
pop_rec = PopularityRecommender()
pop_rec.fit(popularity_df, interactions)

PopularityRecommender fitted on 145,778 artists.


In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.recommenders.hybrid import HybridRecommender

hybrid_rec = HybridRecommender(
    w_popularity=0.1,
    w_svd=0.6,
    w_content=0.3
)
hybrid_rec.fit(pop_rec, svd_rec, cb_rec)

HybridRecommender fitted with engines:
  Popularity  weight: 0.1
  SVD         weight: 0.6
  Content     weight: 0.3


In [5]:

import time

sample_user = interactions["user_id"].iloc[100]

start = time.time()
hybrid_recs = hybrid_rec.recommend(user_id=sample_user, k=10, candidate_pool=100)
print(f"Query time: {time.time()-start:.3f}s")
print(f"\nHybrid Recommendations for user {sample_user[:16]}...")
print(hybrid_recs.to_string(index=False))

Query time: 0.311s

Hybrid Recommendations for user 0000f687d4fe9c1e...
       artist_name  hybrid_score  pop_score_norm  svd_score_norm  cb_score_norm recommendation_source
           shakira      0.765742             0.0        0.790646       0.971181                hybrid
      carlos baute      0.720158             0.0        0.742646       0.915233                hybrid
           baladas      0.600000             0.0        1.000000       0.000000                hybrid
     jenifer lópez      0.600000             0.0        1.000000       0.000000                hybrid
            fergie      0.579692             0.0        0.966154       0.000000                hybrid
   jose el frances      0.552738             0.0        0.921231       0.000000                hybrid
    beatriz luengo      0.544172             0.0        0.906954       0.000000                hybrid
  marcos hernández      0.542843             0.0        0.904738       0.000000                hybrid
          

In [6]:
# Simulate a brand new user with no history
cold_recs = hybrid_rec.recommend(user_id=None, k=10)
print("Cold Start Hybrid Recommendations:")
print(cold_recs[["artist_name", "hybrid_score"]].to_string(index=False))

Cold Start Hybrid Recommendations:
          artist_name  hybrid_score
            radiohead      1.000000
          the beatles      0.989899
             coldplay      0.979798
red hot chili peppers      0.969697
                 muse      0.959596
            metallica      0.949495
           pink floyd      0.939394
          the killers      0.929293
              nirvana      0.919192
          linkin park      0.909091


In [7]:
# How much do results change when you shift weights?
# This is the kind of analysis you'd do before an A/B test in production

weight_configs = [
    (0.1, 0.6, 0.3, "Default (SVD-heavy)"),
    (0.3, 0.5, 0.2, "Balanced"),
    (0.0, 0.8, 0.2, "SVD-only"),
    (0.0, 0.3, 0.7, "Content-heavy"),
    (0.5, 0.3, 0.2, "Popularity-heavy"),
]

for w_pop, w_svd, w_cb, label in weight_configs:
    rec = HybridRecommender(w_popularity=w_pop, w_svd=w_svd, w_content=w_cb)
    rec.fit(pop_rec, svd_rec, cb_rec)
    recs = rec.recommend(user_id=sample_user, k=5)
    print(f"\n{label} [{w_pop}, {w_svd}, {w_cb}]:")
    print(recs["artist_name"].tolist())

HybridRecommender fitted with engines:
  Popularity  weight: 0.1
  SVD         weight: 0.6
  Content     weight: 0.3

Default (SVD-heavy) [0.1, 0.6, 0.3]:
['shakira', 'carlos baute', 'baladas', 'jenifer lópez', 'fergie']
HybridRecommender fitted with engines:
  Popularity  weight: 0.3
  SVD         weight: 0.5
  Content     weight: 0.2

Balanced [0.3, 0.5, 0.2]:
['shakira', 'carlos baute', 'baladas', 'jenifer lópez', 'fergie']
HybridRecommender fitted with engines:
  Popularity  weight: 0.0
  SVD         weight: 0.8
  Content     weight: 0.2

SVD-only [0.0, 0.8, 0.2]:
['shakira', 'jenifer lópez', 'baladas', 'carlos baute', 'fergie']
HybridRecommender fitted with engines:
  Popularity  weight: 0.0
  SVD         weight: 0.3
  Content     weight: 0.7

Content-heavy [0.0, 0.3, 0.7]:
['shakira', 'carlos baute', 'ponto de equilíbrio', 'laurent wolf', 'fantasy']
HybridRecommender fitted with engines:
  Popularity  weight: 0.5
  SVD         weight: 0.3
  Content     weight: 0.2

Popularity-hea

In [8]:
# For the default config, show how each engine contributed to final ranking
print("Score Breakdown — Default Weights [pop=0.1, svd=0.6, cb=0.3]")
print("=" * 80)
print(hybrid_recs[[
    "artist_name", "pop_score_norm", "svd_score_norm",
    "cb_score_norm", "hybrid_score"
]].to_string(index=False))

Score Breakdown — Default Weights [pop=0.1, svd=0.6, cb=0.3]
       artist_name  pop_score_norm  svd_score_norm  cb_score_norm  hybrid_score
           shakira             0.0        0.790646       0.971181      0.765742
      carlos baute             0.0        0.742646       0.915233      0.720158
           baladas             0.0        1.000000       0.000000      0.600000
     jenifer lópez             0.0        1.000000       0.000000      0.600000
            fergie             0.0        0.966154       0.000000      0.579692
   jose el frances             0.0        0.921231       0.000000      0.552738
    beatriz luengo             0.0        0.906954       0.000000      0.544172
  marcos hernández             0.0        0.904738       0.000000      0.542843
              Àëñó             0.0        0.904738       0.000000      0.542843
julio iglesias jr.             0.0        0.904738       0.000000      0.542843


In [9]:
with open("../data/processed/hybrid_recommender.pkl", "wb") as f:
    pickle.dump(hybrid_rec, f)

print("Hybrid recommender saved.")

Hybrid recommender saved.
